# Week 1 · Day 3 — Lab 3
## Inspection & Selection: `loc`, `iloc`, boolean indexing, `query()`

> **AI Engineering Academy** · Gamut Technology Services · pandas 3.x

Before you transform data you must *see* it, then *select* from it precisely.
This lab drills the standard inspection sequence (`head`/`info`/`describe`/`isna`)
and the four selection mechanisms every pandas engineer must keep straight:
**`.loc`** (by label, slice-inclusive), **`.iloc`** (by position, slice-exclusive),
**boolean indexing** (by condition), and **`.query()`** (readable string filters).
It closes on the pandas 3.x rule that trips up everyone: **write values back only
through `.loc[mask, col]`.**

### Learning objectives
1. Inspect a frame with `head`, `info`, `describe`, `isna().sum()`, and `value_counts`.
2. Select by label with `.loc` (inclusive slices) and by position with `.iloc` (exclusive slices).
3. Filter with boolean masks (`&`, `|`, `~`, `isin`, `between`) — parenthesizing each condition.
4. Write readable filters with `.query()`, including `@variable` references.
5. Update values the pandas 3.x way: `df.loc[mask, col] = value` (and know why chained assignment fails).

### Time budget — ~72 min
| Segment | Time |
|---|---|
| Framing & objectives | 5 min |
| **A.** The inspection sequence | 12 min |
| **B.** `.loc` vs `.iloc` | 14 min |
| **C.** Boolean indexing | 15 min |
| **D.** `query()` | 12 min |
| **E.** Updating with `.loc` | 10 min |
| Wrap-up + stretch | 4 min |

### Files you need (in `data/`)
- `users.csv` — 2000 rows; `plan` has a few nulls (for the null-detection steps).
- `events.csv` — 8000 rows; `score` drives the filtering exercises.


In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

print("pandas", pd.__version__)   # target: pandas 3.x on Python 3.13

DATA = Path("data")

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

# Typed ingestion (the Lab 2 habit). plan has a few nulls on purpose.
users = pd.read_csv(
    DATA / "users.csv",
    dtype={"user_id": "int32", "plan": "category", "region": "category"},
    parse_dates=["signup_date"],
)
events = pd.read_csv(
    DATA / "events.csv",
    dtype={"user_id": "int32", "model": "category", "score": "float32"},
    parse_dates=["event_date"],
)
print("users:", users.shape, "| events:", events.shape)

## Part A — The inspection sequence  *(guided)*

Run these four the moment data lands: `head()` (eyeball values), `info()` (shape,
dtypes, non-null counts), `describe()` (numeric summary), `isna().sum()` (nulls per
column). Nulls caught at ingestion are far cheaper than nulls found mid-pipeline.


In [ ]:
users.info()
print("-" * 40)
print(users["plan"].value_counts(dropna=False))

### Exercise A1 — Profile the data
Compute three things: `plan_nulls` (number of null values in `users["plan"]`),
`score_mean` (mean of `events["score"]`, as a float), and `plan_share` (the
**normalized** value counts of `users["plan"]`, i.e. each plan's fraction).


💡 **Hint.** `users["plan"].isna().sum()` for the null count;
`events["score"].mean()`; and `users["plan"].value_counts(normalize=True)` for the
fractions.


In [ ]:
plan_nulls = None    # TODO: count of nulls in users["plan"]
score_mean = None    # TODO: mean event score
plan_share = None    # TODO: normalized value_counts of plan

In [ ]:
check("A1: plan has 20 nulls", lambda: plan_nulls == 20)
check("A1: score_mean is ~73.9", lambda: 70 < score_mean < 78)
check("A1: plan_share sums to 1.0",
      lambda: np.isclose(plan_share.sum(), 1.0))
check("A1: free is the most common plan",
      lambda: plan_share.idxmax() == "free")

## Part B — `.loc` vs `.iloc`

- **`.loc[]`** selects by **label**, and its slices are **inclusive on both ends**.
- **`.iloc[]`** selects by **position** (integer offset), and its slices are
  **exclusive on the right**, like Python lists.

Mixing them up is a classic bug. Match the mechanism to your intent.


In [ ]:
by_id = users.set_index("user_id")
print("loc single label (user 10005):")
print(by_id.loc[10005, ["plan", "region"]])
print("\niloc first 3 rows, first 2 cols (right-exclusive):")
print(by_id.iloc[0:3, 0:2])

### Exercise B1 — Label slice vs position slice
Using `by_id` (indexed by `user_id`):
- `label_slice` = rows for labels `10000` through `10004` **inclusive**, via `.loc`.
- `pos_slice` = the **first 5 rows** via `.iloc`.
Both should have 5 rows. Then set `same_rows` to whether they select the same rows
(they do here, because the index is sorted `10000, 10001, ...`).


💡 **Hint.** `by_id.loc[10000:10004]` is inclusive → 5 rows (10000–10004).
`by_id.iloc[0:5]` is exclusive on the right → rows at positions 0,1,2,3,4.
Compare with `.equals()`.


In [ ]:
label_slice = None   # TODO: by_id.loc[10000:10004]  (inclusive)
pos_slice = None     # TODO: by_id.iloc[0:5]         (first 5 by position)
same_rows = None     # TODO: do they select the same rows?

In [ ]:
check("B1: loc label slice is inclusive -> 5 rows",
      lambda: len(label_slice) == 5)
check("B1: iloc position slice -> 5 rows",
      lambda: len(pos_slice) == 5)
check("B1: here they select identical rows",
      lambda: same_rows is True)

## Part C — Boolean indexing

Pass a boolean Series as the row selector. Combine conditions with `&` (and),
`|` (or), `~` (not) — **wrapping each condition in parentheses**, because `&`
binds tighter than the comparison operators. `isin` tests membership; `between`
tests an inclusive range.


In [ ]:
high = events[events["score"] > 90]
print("score > 90:", len(high), "rows")

pro_like = events[(events["model"] == "atlas-pro") & (events["score"] > 90)]
print("atlas-pro AND score>90:", len(pro_like), "rows")

### Exercise C1 — Compound filters
Build three filtered frames from `events`:
- `top` — score **strictly greater than 90**.
- `mid` — score **between 70 and 85 inclusive** (use `.between`).
- `flagship` — model is `atlas-pro` **or** `orion-8b` (use `.isin`).
Capture each row count into `n_top`, `n_mid`, `n_flagship`.


💡 **Hint.** `events[events["score"] > 90]`;
`events[events["score"].between(70, 85)]`;
`events[events["model"].isin(["atlas-pro", "orion-8b"])]`.


In [ ]:
top = None           # TODO: score > 90
mid = None           # TODO: score between 70 and 85 (inclusive)
flagship = None      # TODO: model in {atlas-pro, orion-8b}
n_top, n_mid, n_flagship = None, None, None

In [ ]:
check("C1: top rows really are all > 90",
      lambda: bool((top["score"] > 90).all()) and n_top == 1025)
check("C1: mid rows are within [70, 85]",
      lambda: bool(top is not None) and bool(mid["score"].between(70, 85).all()) and n_mid == 3215)
check("C1: flagship rows are only the two models",
      lambda: set(flagship["model"].unique()) <= {"atlas-pro", "orion-8b"})

### Exercise C2 — Negation
Build `not_mini`: all events whose model is **not** `atlas-mini`, using `~`. Then
`n_not_mini` is its row count. (There are 8000 events total and 3194 `atlas-mini`.)


In [ ]:
not_mini = None      # TODO: events where model != atlas-mini, using ~
n_not_mini = None

In [ ]:
check("C2: excludes all atlas-mini rows",
      lambda: "atlas-mini" not in set(not_mini["model"].unique()))
check("C2: row count is 8000 - 3194",
      lambda: n_not_mini == 8000 - 3194)

## Part D — `query()`: readable filters

`query()` takes a string expression — often far more readable than stacked boolean
masks, especially in method chains. Reference a Python variable with `@`, and quote
column names containing spaces with backticks. Under the hood it can use `numexpr`
for speed on large frames.


In [ ]:
# Equivalent to events[(events["score"] > 90) & (events["model"] == "atlas-pro")]
q = events.query("score > 90 and model == 'atlas-pro'")
print("query rows:", len(q))

### Exercise D1 — Query with a variable
Set `threshold = 85`. Use `query()` with an `@threshold` reference to select events
scoring above it into `above`. Then, in a single `query`, select events that are
`nova-4` **and** score above the threshold into `nova_high`.


💡 **Hint.** `events.query("score > @threshold")`, and
`events.query("model == 'nova-4' and score > @threshold")`. Note string literals
inside the expression use single quotes.


In [ ]:
threshold = 85
above = None         # TODO: query score > @threshold
nova_high = None     # TODO: query nova-4 AND score > @threshold

In [ ]:
check("D1: above matches the boolean equivalent",
      lambda: len(above) == len(events[events["score"] > threshold]))
check("D1: nova_high is all nova-4",
      lambda: set(nova_high["model"].unique()) == {"nova-4"})
check("D1: nova_high all exceed threshold",
      lambda: bool((nova_high["score"] > threshold).all()))

## Part E — Updating values the pandas 3.x way

Copy-on-Write is always on in pandas 3.0. The **only** reliable way to write values
back is a single `df.loc[mask, col] = value`. **Chained** assignment —
`df[mask][col] = value` — silently fails (it edits a temporary copy) and now warns.


In [ ]:
demo = users.copy()

# The WRONG way (chained): warns and does NOT update demo.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    demo[demo["plan"] == "free"]["region"] = "XX"   # chained assignment
chained_warned = any("ChainedAssignment" in type(w.message).__name__ or
                     "ChainedAssignment" in str(w.category.__name__) for w in caught)
print("chained assignment warned:", chained_warned)
print("did it change anything? XX present:", (demo["region"] == "XX").any())

### Exercise E1 — Rename a plan the correct way
On a copy of `users` called `work`, use a **single `.loc`** assignment to rename
every `"free"` plan to `"basic"`. (Because `plan` is categorical, add the new
category first, or cast to `str` before assigning — the solution casts to `str`.)
Capture `n_basic` = number of `"basic"` rows afterward, and confirm no `"free"`
remains.


💡 **Hint.** Categorical columns only accept existing categories, so
`work["plan"] = work["plan"].astype("str")` first, then
`work.loc[work["plan"] == "free", "plan"] = "basic"`.


In [ ]:
work = users.copy()
# TODO: cast plan to str, then use ONE .loc assignment to set free -> basic
n_basic = None       # TODO: count of "basic" after the update
any_free_left = None # TODO: is any "free" still present?

In [ ]:
check("E1: free was renamed to basic (1203 rows)",
      lambda: n_basic == 1203)
check("E1: no 'free' remains", lambda: any_free_left is False)
check("E1: null plans were left untouched",
      lambda: int(work["plan"].isna().sum()) == 20 or int((work["plan"] == "nan").sum()) == 20)

## Stretch goals *(for fast finishers)*

**S1 — `.iloc` for a shuffled head.** Take `events.sample(frac=1, random_state=0)`
into `shuffled`, then grab its first 10 rows **by position** with `.iloc` into
`sample10`. (Position, not label — the shuffled index labels are meaningless.)

**S2 — Chained query.** In one expression, `query` events for `score > 80`, then
`query` again for `model != 'atlas-mini'`, into `chained_q`. Confirm both
conditions hold.


In [ ]:
# S1
shuffled = None      # TODO: events.sample(frac=1, random_state=0)
sample10 = None      # TODO: first 10 rows of shuffled BY POSITION

# S2
chained_q = None     # TODO: .query("score > 80").query("model != 'atlas-mini'")

In [ ]:
check("S1: sample10 has 10 rows via position", lambda: len(sample10) == 10)
check("S2: chained_q respects both filters",
      lambda: bool((chained_q["score"] > 80).all()) and "atlas-mini" not in set(chained_q["model"].unique()))

## Wrap-up — what you can now do

- Profile a frame with `head`/`info`/`describe`/`isna().sum()`/`value_counts(dropna=False)`.
- Select by label with `.loc` (inclusive) and by position with `.iloc` (right-exclusive).
- Filter with boolean masks (`&`, `|`, `~`, `isin`, `between`), parenthesizing conditions.
- Write readable `query()` filters with `@variable` references.
- Update values correctly with `df.loc[mask, col] = value` and explain why chained assignment fails.

**Next:** Lab 4 — transformation: `assign`, vectorized `.str`/`.dt` ops, `groupby`,
`merge`, and reshaping.
